# Route Factory

> Route factory for job trigger, SSE progress streaming, and cancellation.

In [ ]:
#| default_exp routes.init

In [ ]:
#| export
import asyncio
from typing import Any, Callable, Dict, List, Optional, Tuple
from dataclasses import asdict

from fasthtml.common import Div, Span, Script, APIRouter, FT, EventStream, sse_message

from cjm_fasthtml_interactions.core.state_store import get_session_id
from cjm_workflow_state.state_store import SQLiteWorkflowStateStore

from cjm_fasthtml_job_monitor.html_ids import JobMonitorHtmlIds
from cjm_fasthtml_job_monitor.models import JobMonitorUrls, JobMonitorConfig, ResourceSnapshot
from cjm_fasthtml_job_monitor.services.monitor import JobMonitorService
from cjm_fasthtml_job_monitor.components.trigger import render_job_trigger, render_job_progress_button
from cjm_fasthtml_job_monitor.components.overlay import render_job_overlay, render_job_overlay_placeholder
from cjm_fasthtml_job_monitor.components.modal import (
    render_job_modal, render_sse_connection, render_sse_response,
)

## init_job_monitor_routes

Factory function that creates the three routes (trigger, progress SSE stream, cancel)
and returns them as an `APIRouter` plus a `JobMonitorUrls` for consumers to reference.

### Job Sequence Mode

The `job_args_builder` returns a **list** of `(args, kwargs)` tuples — one per source.
The job monitor submits them sequentially through the `JobQueue`, tracking aggregate
progress across all sources. Single-source is a list of length 1.

### Consumer Setup

The consumer must:
1. Place `render_job_trigger(config, ids, urls)` in their toolbar/action area
2. Place `render_job_overlay_placeholder(ids)` inside the content area to overlay
   (the content area container must have `position: relative`)
3. Place an empty `Div(id=ids.modal)` as a modal placeholder at page level
4. Add `get_sse_headers()` to their app's header list (after HTMX)
5. Provide callbacks for job lifecycle events
6. `job_args_builder` returns `List[Tuple[tuple, dict]]` (list of `(args, kwargs)` pairs)

In [ ]:
#| export
def _get_job_data(service, job_id):
    """Extract job status fields from a Job object."""
    job = service.get_job(job_id)
    if not job:
        return None, 'pending', 0.0, '', None, None
    return (
        job,
        job.status.value if hasattr(job.status, 'value') else str(job.status),
        job.progress or 0.0,
        job.status_message or '',
        job.started_at,
        job.completed_at,
    )

_TERMINAL_STATUSES = {'completed', 'failed', 'cancelled'}


def _get_step_state(state_store, workflow_id, session_id, step_id):
    """Get step state dict from the state store."""
    state = state_store.get_state(workflow_id, session_id)
    return state.get("step_states", {}).get(step_id, {}), state


def _update_step_state(state_store, workflow_id, session_id, step_id, step_state, state):
    """Write step state back to the state store."""
    step_states = state.get("step_states", {})
    step_states[step_id] = step_state
    state["step_states"] = step_states
    state_store.update_state(workflow_id, session_id, state)

In [ ]:
#| export
def init_job_monitor_routes(
    monitor_service: JobMonitorService,           # Service instance
    plugin_name: str,                             # Target plugin for jobs
    state_store: SQLiteWorkflowStateStore,         # For persisting job_id
    workflow_id: str,                             # Workflow ID for state access
    step_id: str,                                 # Step ID for state access
    state_key: str,                               # Key in step state for sequence tracker
    prefix: str,                                  # URL prefix
    overlay_target_id: str,                       # ID of element to overlay
    kb_system_id: Optional[str] = None,           # Keyboard system ID to pause/resume
    on_complete: Optional[Callable] = None,       # async fn(results, request, sess) -> List[FT]
    on_cancel: Optional[Callable] = None,         # async fn(job, request, sess) -> List[FT]
    on_fail: Optional[Callable] = None,           # async fn(job, request, sess) -> List[FT]
    job_args_builder: Optional[Callable] = None,  # fn(state_store, workflow_id, session_id) -> List[(args, kwargs)]
    config: Optional[JobMonitorConfig] = None,    # UI config
    id_prefix: str = "jm",                        # HTML ID prefix
    icon_fn: Optional[Callable] = None,           # Icon renderer fn(name, **kwargs) -> FT
) -> Tuple[APIRouter, JobMonitorUrls, JobMonitorHtmlIds]:  # (router, urls, ids)
    """Initialize job monitor routes with SSE-based progress streaming.
    
    Supports job sequences: `job_args_builder` returns a list of (args, kwargs)
    tuples. Jobs are submitted sequentially, with aggregate progress tracking.
    Single-source is a list of length 1.
    
    `on_complete` receives a list of job result objects (one per source).
    """
    config = config or JobMonitorConfig()
    ids = JobMonitorHtmlIds(prefix=id_prefix)

    router = APIRouter(prefix=prefix)

    # --- Sequence state helpers ---
    # The sequence tracker stored in step_state[state_key] is a dict:
    #   {"job_id": str, "source_index": int, "total": int, "results": [...],
    #    "remaining_args": [{"args": [...], "kwargs": {...}}, ...]}
    # For single jobs, total=1. When no sequence is active, the key is absent.

    def _get_seq(session_id):
        """Get the sequence tracker from state, or None."""
        step_state, _ = _get_step_state(state_store, workflow_id, session_id, step_id)
        return step_state.get(state_key)

    def _set_seq(session_id, seq):
        """Store the sequence tracker in state."""
        step_state, state = _get_step_state(state_store, workflow_id, session_id, step_id)
        step_state[state_key] = seq
        _update_step_state(state_store, workflow_id, session_id, step_id, step_state, state)

    def _clear_seq(session_id):
        """Remove the sequence tracker from state."""
        step_state, state = _get_step_state(state_store, workflow_id, session_id, step_id)
        step_state.pop(state_key, None)
        _update_step_state(state_store, workflow_id, session_id, step_id, step_state, state)

    @router
    async def trigger(request, sess):
        """Submit a job sequence and return the modal + overlay + progress button via OOB."""
        session_id = get_session_id(sess)

        # Build job args list from consumer callback
        args_list = [((), {})]
        if job_args_builder:
            args_list = job_args_builder(state_store, workflow_id, session_id)

        if not args_list:
            return ()

        total = len(args_list)

        # Submit the first job
        first_args, first_kwargs = args_list[0]
        job_id = await monitor_service.submit_job(plugin_name, *first_args, **first_kwargs)

        # Store sequence tracker (includes the full args list for remaining sources)
        seq = {
            "job_id": job_id,
            "source_index": 0,
            "total": total,
            "results": [],
            "remaining_args": [
                {"args": list(a), "kwargs": k} for a, k in args_list[1:]
            ],
        }
        _set_seq(session_id, seq)

        # Build URLs
        urls = JobMonitorUrls(
            trigger=trigger.to(),
            progress=progress.to(),
            cancel=cancel.to(),
        )

        # Get initial job data
        _, status, prog_val, msg, started, completed = _get_job_data(monitor_service, job_id)
        if total > 1:
            msg = f"Source 1/{total}: {msg}" if msg else f"Source 1/{total}: Starting..."
        logs = monitor_service.get_logs(plugin_name, config.log_lines)
        resources = monitor_service.get_resource_snapshot(plugin_name)

        # Build OOB responses
        oob_parts = []

        # 1. Progress button replaces trigger slot
        prog_btn = render_job_progress_button(config, ids)
        prog_btn.attrs['hx-swap-oob'] = "true"
        oob_parts.append(prog_btn)

        # 2. Overlay replaces overlay placeholder
        overlay = render_job_overlay(ids, config)
        overlay.attrs['hx-swap-oob'] = "true"
        oob_parts.append(overlay)

        # 3. Modal with SSE connection (session_id as stream identifier)
        modal_el = render_job_modal(
            config, ids, urls, job_id=session_id,  # SSE stream keyed by session
            status=status, progress_value=prog_val,
            status_message=msg, started_at=started, completed_at=completed,
            logs=logs, resources=resources, open_on_render=True,
        )
        modal_el.attrs['hx-swap-oob'] = "true"
        oob_parts.append(modal_el)

        # 4. Keyboard pause script
        if kb_system_id:
            oob_parts.append(Script(f"window.kbCoordinator?.pause('{kb_system_id}');"))

        return tuple(oob_parts)

    @router
    async def progress(request, sess, job_id: str = ''):
        """SSE stream endpoint — pushes progress for a job sequence until all complete."""
        session_id = get_session_id(sess)

        # Build URLs
        urls = JobMonitorUrls(
            trigger=trigger.to(),
            progress=progress.to(),
            cancel=cancel.to(),
        )

        def _build_cleanup_oob():
            """Build the OOB elements for sequence completion/cancel/fail."""
            oob = []
            # Remove overlay
            overlay_ph = render_job_overlay_placeholder(ids)
            overlay_ph.attrs['hx-swap-oob'] = "true"
            oob.append(overlay_ph)
            # Restore trigger button
            trigger_btn = render_job_trigger(config, ids, urls, icon_fn=icon_fn)
            trigger_btn.attrs['hx-swap-oob'] = "true"
            oob.append(trigger_btn)
            # Kill SSE connection
            inert_sse = render_sse_connection(ids, urls, job_id='', is_active=False)
            inert_sse.attrs['hx-swap-oob'] = "true"
            oob.append(inert_sse)
            # Resume keyboard
            if kb_system_id:
                oob.append(Script(f"window.kbCoordinator?.resume('{kb_system_id}');"))
            return oob

        async def sse_stream():
            try:
                prev_logs = None
                resource_cycle = 0

                while True:
                    # Get current sequence state
                    seq = _get_seq(session_id)
                    if not seq:
                        yield sse_message(render_sse_response(ids, urls))
                        return

                    current_job_id = seq["job_id"]
                    source_idx = seq["source_index"]
                    total = seq["total"]
                    show_source = total > 1

                    # Get current job data
                    job, status, prog_val, msg, started, completed = _get_job_data(
                        monitor_service, current_job_id
                    )

                    if not job:
                        yield sse_message(render_sse_response(ids, urls))
                        return

                    # Prefix message with source context
                    display_msg = msg
                    if show_source:
                        src_label = f"Source {source_idx + 1}/{total}"
                        display_msg = f"{src_label}: {msg}" if msg else f"{src_label}: Processing..."

                    # Get logs and resources (selective cadence)
                    logs = monitor_service.get_logs(plugin_name, config.log_lines)
                    resources = None
                    if resource_cycle == 0:
                        resources = monitor_service.get_resource_snapshot(plugin_name)
                    resource_cycle = (resource_cycle + 1) % 4

                    # Check for terminal state on current job
                    if status in _TERMINAL_STATUSES:
                        if status == 'completed':
                            # Store result and check if more sources remain
                            seq["results"].append(job.result)
                            remaining = seq.get("remaining_args", [])

                            if remaining:
                                # Submit next source
                                next_entry = remaining.pop(0)
                                next_args = tuple(next_entry["args"])
                                next_kwargs = next_entry["kwargs"]
                                next_job_id = await monitor_service.submit_job(
                                    plugin_name, *next_args, **next_kwargs
                                )
                                seq["job_id"] = next_job_id
                                seq["source_index"] = source_idx + 1
                                seq["remaining_args"] = remaining
                                _set_seq(session_id, seq)

                                # Push update showing transition to next source
                                next_msg = f"Source {source_idx + 2}/{total}: Starting..."
                                yield sse_message(render_sse_response(
                                    ids, urls, status='running', progress_value=0.0,
                                    status_message=next_msg, started_at=None,
                                    logs=logs if logs != prev_logs else None,
                                    resources=resources,
                                ))
                                prev_logs = logs
                                await asyncio.sleep(config.sse_interval_s)
                                continue  # Continue outer loop with new job

                            # All sources complete — fire on_complete with results list
                            _clear_seq(session_id)
                            resources = monitor_service.get_resource_snapshot(plugin_name)
                            extra_oob = []
                            if on_complete:
                                result = await on_complete(seq["results"], request, sess)
                                if result:
                                    extra_oob.extend(
                                        result if isinstance(result, (list, tuple)) else [result]
                                    )
                            extra_oob.extend(_build_cleanup_oob())

                            final_msg = f"Completed ({total} source{'s' if total > 1 else ''})"
                            yield sse_message(render_sse_response(
                                ids, urls, status='completed', progress_value=1.0,
                                status_message=final_msg, started_at=started,
                                completed_at=completed, logs=logs, resources=resources,
                                include_footer=True, extra_oob=extra_oob,
                            ))
                            return

                        else:
                            # Failed or cancelled — abort entire sequence
                            _clear_seq(session_id)
                            resources = monitor_service.get_resource_snapshot(plugin_name)
                            extra_oob = []
                            if status == 'cancelled' and on_cancel:
                                result = await on_cancel(job, request, sess)
                                if result:
                                    extra_oob.extend(
                                        result if isinstance(result, (list, tuple)) else [result]
                                    )
                            elif status == 'failed' and on_fail:
                                result = await on_fail(job, request, sess)
                                if result:
                                    extra_oob.extend(
                                        result if isinstance(result, (list, tuple)) else [result]
                                    )
                            extra_oob.extend(_build_cleanup_oob())

                            yield sse_message(render_sse_response(
                                ids, urls, status=status, progress_value=prog_val,
                                status_message=display_msg, started_at=started,
                                completed_at=completed, logs=logs, resources=resources,
                                include_footer=True, extra_oob=extra_oob,
                            ))
                            return

                    # Still running — push selective update
                    yield sse_message(render_sse_response(
                        ids, urls, status=status, progress_value=prog_val,
                        status_message=display_msg, started_at=started,
                        completed_at=completed,
                        logs=logs if logs != prev_logs else None,
                        resources=resources,
                    ))
                    prev_logs = logs
                    await asyncio.sleep(config.sse_interval_s)

            except Exception as e:
                print(f"[JobMonitor SSE] Error in stream: {e}")
                import traceback
                traceback.print_exc()
                return

        return EventStream(sse_stream())

    @router
    async def cancel(request, sess):
        """Cancel the active job sequence."""
        session_id = get_session_id(sess)
        seq = _get_seq(session_id)
        if seq:
            await monitor_service.cancel_job(seq["job_id"])
        # Cancellation is async — the SSE stream detects the terminal state
        return ""

    # Build URLs
    urls = JobMonitorUrls(
        trigger=trigger.to(),
        progress=progress.to(),
        cancel=cancel.to(),
    )

    return router, urls, ids

## Resume Helper

Utility for consumers to check for in-flight jobs on page load.
Returns appropriate UI state (trigger button vs progress button, overlay vs placeholder).

In [ ]:
#| export
def check_inflight_job(
    monitor_service: JobMonitorService,       # Service instance
    plugin_name: str,                         # Target plugin
    state_store: SQLiteWorkflowStateStore,    # State store
    workflow_id: str,                         # Workflow ID
    session_id: str,                          # Session ID
    step_id: str,                             # Step ID
    state_key: str,                           # State key for sequence tracker
    config: JobMonitorConfig,                 # Display config
    ids: JobMonitorHtmlIds,                   # Element IDs
    urls: JobMonitorUrls,                     # Route URLs
    icon_fn: Optional[Callable] = None,       # Icon renderer
) -> Tuple[Optional[FT], Optional[FT], Optional[FT], bool]:
    """Check for in-flight job sequence and return appropriate UI state.
    
    The state_key stores a sequence tracker dict with a "job_id" field.
    
    Returns:
        - Button element (trigger or progress button)
        - Overlay element (active overlay or empty placeholder)
        - Modal element (with SSE connection if running, or empty placeholder)
        - Whether a job is currently running
    """
    state = state_store.get_state(workflow_id, session_id)
    step_states = state.get("step_states", {})
    step_state = step_states.get(step_id, {})
    seq = step_state.get(state_key)

    modal_placeholder = Div(id=ids.modal)

    if not seq or not isinstance(seq, dict):
        # No in-flight sequence
        return (
            render_job_trigger(config, ids, urls, icon_fn=icon_fn),
            render_job_overlay_placeholder(ids),
            modal_placeholder,
            False,
        )

    job_id = seq.get("job_id")
    if not job_id:
        # Malformed tracker — clean up
        step_state.pop(state_key, None)
        step_states[step_id] = step_state
        state["step_states"] = step_states
        state_store.update_state(workflow_id, session_id, state)
        return (
            render_job_trigger(config, ids, urls, icon_fn=icon_fn),
            render_job_overlay_placeholder(ids),
            modal_placeholder,
            False,
        )

    # Check current job status
    job, status, prog_val, msg, started, completed = _get_job_data(monitor_service, job_id)

    if not job or status in _TERMINAL_STATUSES:
        # Job finished — clean up state
        step_state.pop(state_key, None)
        step_states[step_id] = step_state
        state["step_states"] = step_states
        state_store.update_state(workflow_id, session_id, state)

        return (
            render_job_trigger(config, ids, urls, icon_fn=icon_fn),
            render_job_overlay_placeholder(ids),
            modal_placeholder,
            False,
        )

    # Job is still running — return modal with SSE connection for resume
    total = seq.get("total", 1)
    source_idx = seq.get("source_index", 0)
    if total > 1:
        msg = f"Source {source_idx + 1}/{total}: {msg}" if msg else f"Source {source_idx + 1}/{total}: Processing..."

    logs = monitor_service.get_logs(plugin_name, config.log_lines)
    resources = monitor_service.get_resource_snapshot(plugin_name)

    modal_el = render_job_modal(
        config, ids, urls, job_id=session_id,  # SSE stream keyed by session
        status=status, progress_value=prog_val,
        status_message=msg, started_at=started, completed_at=completed,
        logs=logs, resources=resources,
        open_on_render=False,  # Don't auto-open on page load
    )

    return (
        render_job_progress_button(config, ids),
        render_job_overlay(ids, config),
        modal_el,
        True,
    )

In [ ]:
# Test URL generation
from cjm_fasthtml_job_monitor.models import JobMonitorUrls

# Verify URLs are constructed correctly
urls = JobMonitorUrls(
    trigger="/workflow/seg/fa/trigger",
    progress="/workflow/seg/fa/progress",
    cancel="/workflow/seg/fa/cancel",
)
assert urls.trigger.endswith("/trigger")
assert urls.progress.endswith("/progress")
assert urls.cancel.endswith("/cancel")
print(f"URLs: {urls}")
print("Route factory URL construction: OK")

In [ ]:
# Test check_inflight_job with no active job
from cjm_fasthtml_job_monitor.html_ids import JobMonitorHtmlIds
from cjm_fasthtml_job_monitor.models import JobMonitorConfig

class MockStateStore:
    def __init__(self, state=None):
        self._state = state or {}
    def get_state(self, wf_id, sess_id):
        return dict(self._state)
    def update_state(self, wf_id, sess_id, state):
        self._state = state

ids = JobMonitorHtmlIds(prefix="jm")
config = JobMonitorConfig()
urls = JobMonitorUrls(trigger="/trigger", progress="/progress", cancel="/cancel")
store = MockStateStore({"step_states": {"seg": {}}})

btn_el, overlay_el, modal_el, is_running = check_inflight_job(
    monitor_service=None,  # Not needed when no sequence in state
    plugin_name="test-plugin",
    state_store=store,
    workflow_id="wf1",
    session_id="s1",
    step_id="seg",
    state_key="fa_job_id",
    config=config,
    ids=ids,
    urls=urls,
)
assert not is_running
assert btn_el.attrs['id'] == 'jm-trigger-slot'
assert overlay_el.attrs['id'] == 'jm-overlay'
assert len(overlay_el.children) == 0  # Placeholder
assert modal_el.attrs['id'] == 'jm-modal'
assert modal_el.tag == 'div'  # Empty placeholder, not dialog
print("check_inflight_job (no active job): OK")

# Test with malformed tracker (not a dict)
store2 = MockStateStore({"step_states": {"seg": {"fa_job_id": "some-old-string"}}})
btn2, overlay2, modal2, running2 = check_inflight_job(
    monitor_service=None, plugin_name="test-plugin",
    state_store=store2, workflow_id="wf1", session_id="s1",
    step_id="seg", state_key="fa_job_id",
    config=config, ids=ids, urls=urls,
)
assert not running2  # Old string format treated as no active job
print("check_inflight_job (legacy string state): OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()